<a href="https://colab.research.google.com/github/atharvsalokhe30/Deep_Learning-_project_practice/blob/main/BreastCancerPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

df = pd.read_csv('/content/breastcancer.csv')
display(df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/breastcancer.csv'

### Data Preprocessing - Initial Inspection

First, let's get a summary of the DataFrame, including data types and non-null values, to identify any immediate issues like missing data or incorrect data types.

In [ ]:
display(df.info())

Next, let's look at the descriptive statistics for the numerical columns to understand their distribution, central tendency, and spread. This can help identify outliers or unusual patterns.

In [ ]:
display(df.describe())

The 'diagnosis' column appears to be a categorical target variable. Let's examine its unique values and their counts to understand the class distribution.

In [ ]:
display(df['diagnosis'].value_counts())

### Preprocessing Steps

1.  **Drop 'id' column**: This column is a unique identifier and does not contribute to the predictive power of the model.
2.  **Encode 'diagnosis' column**: Convert the categorical 'diagnosis' column ('M' and 'B') into numerical values (e.g., 1 for 'Malignant' and 0 for 'Benign').
3.  **Feature Scaling**: Standardize the numerical features to ensure that all features contribute equally to the model, preventing features with larger values from dominating the learning process.

In [ ]:
# Drop the 'id' column
df_processed = df.drop('id', axis=1)

# Encode the 'diagnosis' column
df_processed['diagnosis'] = df_processed['diagnosis'].map({'M': 1, 'B': 0})

# Separate features (X) and target (y)
X = df_processed.drop('diagnosis', axis=1)
y = df_processed['diagnosis']

display(f"Shape of features (X): {X.shape}")
display(f"Shape of target (y): {y.shape}")
display(X.head())
display(y.head())

#### Feature Scaling

Now, let's apply `StandardScaler` to our features `X` to standardize them. This transforms the data so that it has a mean of 0 and a standard deviation of 1.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Convert the scaled features back to a DataFrame for better readability
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

display(X_scaled_df.head())

### Model Training - Support Vector Machine (SVM)

Now that the data is preprocessed, let's split it into training and testing sets and then train a classification model. We'll use an SVM classifier for this task.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled_df, y, test_size=0.3, random_state=42)

display(f"X_train shape: {X_train.shape}")
display(f"X_test shape: {X_test.shape}")
display(f"y_train shape: {y_train.shape}")
display(f"y_test shape: {y_test.shape}")

#### Train the SVM Model

Next, we'll initialize and train the Support Vector Machine (SVM) classifier using the training data.

In [ ]:
svm_model = SVC(random_state=42)
svm_model.fit(X_train, y_train)

display("SVM model trained successfully.")

### Model Evaluation

Now, let's evaluate the trained SVM model's performance on the test set (`X_test`, `y_test`). We will predict the labels for the test data and then calculate the accuracy score and display a classification report.

In [ ]:
# Make predictions on the test set
y_pred = svm_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
display(f"Model Accuracy: {accuracy:.4f}")

# Display classification report
classification_rep = classification_report(y_test, y_pred)
display("Classification Report:")
display(classification_rep)

### Explanation of Label Encoding

**Label Encoding** is a technique used to convert categorical labels into numerical format. In our dataset, we used it to transform the 'diagnosis' column from 'M' (Malignant) and 'B' (Benign) into `1` and `0` respectively.

#### Why is Label Encoding Necessary?

1.  **Machine Learning Algorithms Require Numerical Input**: Most machine learning algorithms cannot directly work with categorical data (text labels). They operate on numerical values. Label encoding provides a way to represent these categories as numbers.

2.  **Preserving Ordinality (When Applicable)**: If the categories have an inherent order (e.g., 'low', 'medium', 'high'), label encoding assigns numerical values that maintain this order. While 'M' and 'B' don't have a strict ordinal relationship in terms of magnitude, converting them to 0 and 1 allows the model to differentiate between the two classes. For binary classification problems like ours, this is a straightforward and effective approach.

3.  **Simplicity**: For target variables with a small number of unique categories, especially binary ones, label encoding is simple and computationally efficient.

In our case, encoding 'M' as 1 and 'B' as 0 makes the `diagnosis` column suitable for our classification model to predict whether a tumor is malignant or benign.

### Artificial Neural Network (ANN) Model

Now, let's proceed to build, train, and evaluate an Artificial Neural Network (ANN) model using our preprocessed and scaled dataset. We'll use TensorFlow/Keras for this.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Define the ANN model
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid') # Output layer for binary classification
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
model.summary()

#### Train the ANN Model

Now, let's train the ANN model using our `X_train` and `y_train` data. We'll use a validation split to monitor performance on unseen data during training.

In [ ]:
# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

#### Evaluate the ANN Model

Finally, let's evaluate the trained ANN model on the test set (`X_test`, `y_test`) to assess its generalization performance.

In [ ]:
# Evaluate the model on the test set
loss, accuracy_ann = model.evaluate(X_test, y_test, verbose=0)

display(f"ANN Model Test Loss: {loss:.4f}")
display(f"ANN Model Test Accuracy: {accuracy_ann:.4f}")

# Make predictions (probabilities)
y_pred_ann_prob = model.predict(X_test)
# Convert probabilities to binary predictions (0 or 1)
y_pred_ann = (y_pred_ann_prob > 0.5).astype(int)

# Display classification report for ANN
classification_rep_ann = classification_report(y_test, y_pred_ann)
display("ANN Classification Report:")
display(classification_rep_ann)

### Feature Scaling: Normalization vs. Standardization for ANNs

**Feature scaling** is a method used to normalize the range of independent variables or features of data. It's a critical preprocessing step for many machine learning algorithms, and especially for Artificial Neural Networks (ANNs).

There are two common types of scaling:

1.  **Normalization (Min-Max Scaling)**: This technique rescales the features to a fixed range, usually between 0 and 1. The formula is:
    `X_normalized = (X - X_min) / (X_max - X_min)`

2.  **Standardization (Z-score Normalization)**: This technique transforms the data to have a mean of 0 and a standard deviation of 1. The formula is:
    `X_standardized = (X - μ) / σ` (where μ is the mean and σ is the standard deviation).

#### How Scaling Improves ANN Performance and Convergence:

1.  **Faster Convergence**: ANNs often use gradient-based optimization algorithms (like Gradient Descent, Adam, etc.) to update weights. When features have vastly different scales, the cost function can become elongated and skewed. This makes it harder for the optimizer to find the minimum efficiently, leading to slow convergence or getting stuck in local minima. Scaling the features ensures that all features contribute approximately equally to the gradient, making the optimization landscape more spherical and allowing the optimizer to take more direct paths to the minimum, thus **accelerating convergence**.

2.  **Prevents Dominance by Features with Larger Values**: Without scaling, features with larger numerical ranges might disproportionately influence the loss function and the learning process. The model might perceive these features as more important simply because of their magnitude, even if they are not inherently more informative. Scaling ensures that **all features are treated equally** based on their underlying patterns rather than their raw scale.

3.  **Better Weight Initialization**: Many weight initialization techniques for neural networks assume that input data is standardized or normalized. If the input data is not scaled, these initialization strategies might not work as effectively, potentially leading to vanishing or exploding gradients during early training phases.

4.  **Regularization Benefits**: Some regularization techniques (e.g., L1/L2 regularization) are sensitive to feature scales. If features are not scaled, the regularization penalty might be applied unevenly, benefiting features with smaller scales while penalizing those with larger scales more heavily.

5.  **Activation Functions Performance**: Many activation functions, particularly sigmoid and tanh, are sensitive to the range of input values. If inputs are too large, these functions can saturate (output values close to their maximum or minimum), causing gradients to become very small (vanishing gradient problem), which halts learning. Scaling keeps inputs within a range where these activation functions are most responsive.

In our case, we applied **Standardization** using `StandardScaler` to our feature set (`X`) before training both the SVM and the ANN. This step was crucial and likely contributed significantly to the high accuracy and efficient training observed in both models.

### Confusion Matrix and Classification Report

We have already generated the classification reports for both the SVM and ANN models. Now, let's visualize their performance using **Confusion Matrices**.

A confusion matrix is a table that is often used to describe the performance of a classification model (or "classifier") on a set of test data for which the true values are known. It allows the visualization of the performance of an algorithm. Each row of the matrix represents the instances in an actual class, while each column represents the instances in a predicted class.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# --- Confusion Matrix for SVM Model ---
cm_svm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Benign (0)', 'Malignant (1)'],
            yticklabels=['Benign (0)', 'Malignant (1)']) # Updated labels
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for SVM Model')
plt.show()

display("SVM Classification Report:")
display(classification_rep)

In [ ]:
# --- Confusion Matrix for ANN Model ---
cm_ann = confusion_matrix(y_test, y_pred_ann)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_ann, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Benign (0)', 'Malignant (1)'],
            yticklabels=['Benign (0)', 'Malignant (1)']) # Updated labels
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for ANN Model')
plt.show()

display("ANN Classification Report:")
display(classification_rep_ann)